# Data Profiling e Integración de Fuentes para Clustering de Empresas

## Objetivo

Este cuaderno documenta el proceso de análisis, validación e integración de dos fuentes de datos empresariales con el objetivo de construir un dataset adecuado para técnicas de clustering.

## Fuentes de datos

- Dataset 1: Leads provenientes de CRM
- Dataset 2: Registro de horas trabajadas por empresa

## Enfoque

Se sigue una metodología basada en:

1. Data profiling
2. Validación de hipótesis de integración
3. Evaluación de consistencia entre fuentes
4. Toma de decisiones basada en evidencia
5. Recomendación de siguiente paso (staging y estrategia de integración)

## 1. Imports y configuración


In [10]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


## 2. Configuración de archivos


In [ ]:
LEADS_FILE = 'leads.xlsx'
HORAS_FILE = 'proyectos_empresa.xlsx'

N_FILAS_PRUEBA = None


## 3. Función de carga de datos


In [12]:
def leer_archivo(path, nrows=None):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == '.csv':
        return pd.read_csv(path, nrows=nrows)
    elif suffix in ['.xlsx', '.xls']:
        return pd.read_excel(path, nrows=nrows)
    else:
        raise ValueError(f'Formato no soportado: {suffix}')


## 4. Carga de datasets


In [13]:
df_leads = leer_archivo(LEADS_FILE, nrows=N_FILAS_PRUEBA)
df_horas = leer_archivo(HORAS_FILE, nrows=N_FILAS_PRUEBA)

print('Leads:', df_leads.shape)
print('Horas:', df_horas.shape)


Leads: (440, 15)
Horas: (412, 18)


C:\Users\asus\AppData\Roaming\Python\Python312\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


## 4.b Perfilado rápido (calidad y esquema)


In [14]:
def _profile_table(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        'columna': df.columns,
        'dtype': [str(df[c].dtype) for c in df.columns],
        'n_null': [int(df[c].isna().sum()) for c in df.columns],
        'pct_null': [float(df[c].isna().mean() * 100) for c in df.columns],
        'n_unique': [int(df[c].nunique(dropna=True)) for c in df.columns],
    })
    return out.sort_values(['pct_null', 'n_unique'], ascending=[False, True]).reset_index(drop=True)

def _normalize_for_compare(value) -> str:
    if value is None:
        return ''
    if isinstance(value, float) and np.isnan(value):
        return ''
    s = str(value).strip()
    if not s:
        return ''
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    s = s.upper()
    s = re.sub(r'[^A-Z0-9 ]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def _company_quality(df: pd.DataFrame, col: str, label: str) -> None:
    if col not in df.columns:
        print(f'[{label}] Columna no encontrada: {col}')
        return
    s = df[col]
    s_str = s.astype('string')
    blank = s_str.isna() | (s_str.str.strip() == '')
    print(f'\n[{label}] Calidad de {col}:')
    print('- filas:', len(df))
    print('- n_null:', int(s.isna().sum()))
    print('- n_blank (null o vacío):', int(blank.sum()))
    print('- n_unique (no-null):', int(s.nunique(dropna=True)))
    print('- n_duplicadas (por valor no-null):', int(s.dropna().duplicated().sum()))

    top = (s_str.fillna('')
              .map(lambda x: x.strip())
              .replace('', pd.NA)
              .dropna()
              .value_counts()
              .head(15)
              .reset_index())
    if len(top):
        top.columns = [col, 'freq']
        display(top)

    # Normalización rápida para estimar colisiones
    norm = s_str.map(_normalize_for_compare)
    norm = norm.replace('', pd.NA).dropna()
    if len(norm):
        print('- n_unique_norm:', int(norm.nunique()))
        print('- colisiones_por_norm (valores distintos que caen al mismo norm):', int(norm.duplicated().sum()))

def _exact_match_summary() -> None:
    if 'Company' not in df_leads.columns or 'EMPRESA' not in df_horas.columns:
        print('\n[JOIN] No se puede calcular resumen: falta Company o EMPRESA.')
        return

    leads_raw = (df_leads['Company'].astype('string').fillna('').map(lambda x: x.strip()))
    horas_raw = (df_horas['EMPRESA'].astype('string').fillna('').map(lambda x: x.strip()))
    leads_raw = leads_raw.replace('', pd.NA).dropna()
    horas_raw = horas_raw.replace('', pd.NA).dropna()

    raw_matches = sorted(set(leads_raw).intersection(set(horas_raw)))
    print('\n[JOIN] Coincidencias exactas (sin normalizar):', len(raw_matches))
    if len(raw_matches):
        print('  Muestra:', raw_matches[:20])

    leads_norm = leads_raw.map(_normalize_for_compare).replace('', pd.NA).dropna()
    horas_norm = horas_raw.map(_normalize_for_compare).replace('', pd.NA).dropna()
    norm_matches = sorted(set(leads_norm).intersection(set(horas_norm)))
    print('[JOIN] Coincidencias exactas (normalizadas):', len(norm_matches))
    if len(norm_matches):
        print('  Muestra:', norm_matches[:20])

print('--- Perfilado LEADS ---')
display(_profile_table(df_leads))

print('\n--- Perfilado HORAS ---')
display(_profile_table(df_horas))

# Calidad de las llaves candidatas para join
_company_quality(df_leads, 'Company', 'LEADS')
_company_quality(df_horas, 'EMPRESA', 'HORAS')

# Evidencia cuantitativa: ¿hay intersección exacta?
_exact_match_summary()

--- Perfilado LEADS ---


,columna,dtype,n_null,pct_null,n_unique
0,First Name,float64,440,100.000000,0
1,Last Name,float64,440,100.000000,0
2,Email,float64,440,100.000000,0
3,Phone,float64,440,100.000000,0
4,Mobile,float64,440,100.000000,0
5,Website,float64,440,100.000000,0
6,No. of Employees,float64,440,100.000000,0
7,Annual Revenue,float64,440,100.000000,0
8,Linkedin,float64,440,100.000000,0
9,País.,object,415,94.318182,8



--- Perfilado HORAS ---


,columna,dtype,n_null,pct_null,n_unique
0,AVANCE_REAL,float64,412,100.000000,0
1,AVANCE_ESTIMADO,float64,412,100.000000,0
2,ID_COL_RESPONSABLE,float64,379,91.990291,7
3,FACTURACION,float64,235,57.038835,120
4,HORAS_ESTIMADAS,float64,187,45.388350,107
5,HORAS_EJECUTADAS_FACTURABLES,float64,26,6.310680,314
6,HORAS_EJECUTADAS,float64,20,4.854369,319
7,FECHA_CORTE,datetime64[ns],19,4.611650,5
8,EN_EJECUCION,float64,4,0.970874,2
9,MOSTRAR_LISTAS,int64,0,0.000000,2



[LEADS] Calidad de Company:
- filas: 440
- n_null: 0
- n_blank (null o vacío): 0
- n_unique (no-null): 330
- n_duplicadas (por valor no-null): 110


,Company,freq
0,GRUPO ALEN,10
1,PEPSICO,7
2,GRUPO BIMBO,7
3,WALMART,7
4,CASA CUERVO,5
5,BACARDI,5
6,SEGUROS MONTERREY NEW YORK LIFE,5
7,LAMOSA,5
8,UNILEVER,4
9,AIG,4


- n_unique_norm: 328
- colisiones_por_norm (valores distintos que caen al mismo norm): 112

[HORAS] Calidad de EMPRESA:
- filas: 412
- n_null: 0
- n_blank (null o vacío): 0
- n_unique (no-null): 120
- n_duplicadas (por valor no-null): 292


,EMPRESA,freq
0,Corporacion GPF,66
1,Corporación Maresa,22
2,FPA,21
3,Veolia Latam,21
4,Aseguradora del Sur,20
5,Intaco,12
6,La Fabril,10
7,Telefónica EC,10
8,NOVA Ecuador,10
9,"Millicom - Telefónica PA, NI",9


- n_unique_norm: 119
- colisiones_por_norm (valores distintos que caen al mismo norm): 293

[JOIN] Coincidencias exactas (sin normalizar): 0
[JOIN] Coincidencias exactas (normalizadas): 0


## 5. Validación de hipótesis de integración

### Hipótesis

Se plantea que:

> Las empresas presentes en el dataset de leads deberían coincidir con las empresas presentes en el dataset de horas trabajadas.

### Objetivo

Validar si es posible realizar un join confiable entre ambas fuentes.


## 6. Evaluación detallada (coincidencias exactas normalizadas y aproximadas) (opcional)

El resumen de la sección **4.b (Perfilado rápido)** puede ampliarse con una rutina más detallada de normalización y similitud.
Esta sección se conserva porque aporta evidencia técnica adicional:

- normaliza nombres con eliminación de acentos,
- reduce ruido por sufijos legales comunes,
- calcula coincidencias exactas después de normalizar,
- y genera sugerencias por similitud para evidenciar que no existen matches confiables.

Esto respalda formalmente la decisión de **no forzar un join** entre ambas fuentes usando solo el nombre de la empresa.

In [15]:
import re
import unicodedata
import pandas as pd

def _strip_accents(s: str) -> str:
    return ''.join(ch for ch in unicodedata.normalize('NFKD', s) if not unicodedata.combining(ch))

def _normalize_name(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip()
    s = _strip_accents(s)
    s = s.upper()
    # quitar caracteres raros, dejar letras/números/espacios
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)
    # quitar sufijos legales comunes
    s = re.sub(r"\b(SA|S A|S\.A|S\.A\.S|SAS|LTDA|CIA|CORP|INC|LLC|DE|DEL|LA|EL)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Tomar listas directamente desde los DataFrames (evita dependencia de secciones eliminadas)
if 'Company' not in df_leads.columns or 'EMPRESA' not in df_horas.columns:
    print("No se puede comparar: falta Company o EMPRESA en los dataframes.")
else:
    companies = (df_leads['Company'].astype('string').fillna('')
                .map(lambda x: x.strip())
                .replace('', pd.NA)
                .dropna()
                .unique()
                .tolist())
    empresas = (df_horas['EMPRESA'].astype('string').fillna('')
                .map(lambda x: x.strip())
                .replace('', pd.NA)
                .dropna()
                .unique()
                .tolist())

    if not companies or not empresas:
        print("No hay valores suficientes para comparar (listas vacías).")
    else:
        comp_norm = {c: _normalize_name(c) for c in companies}
        emp_norm = {e: _normalize_name(e) for e in empresas}

        # 1) Coincidencias exactas después de normalizar
        inv_emp = {}
        for e, ne in emp_norm.items():
            inv_emp.setdefault(ne, []).append(e)

        exact_matches = []
        for c, nc in comp_norm.items():
            for e in inv_emp.get(nc, []):
                exact_matches.append({
                    "Company": c,
                    "EMPRESA": e,
                    "tipo": "exact_norm",
                    "score": 100
                })

        df_exact = pd.DataFrame(exact_matches)
        print(f"Coincidencias EXACTAS (tras normalizar): {len(df_exact)}")
        if len(df_exact):
            display(df_exact.sort_values(['Company', 'EMPRESA']).reset_index(drop=True))

        # 2) Coincidencias por similitud (fuzzy). Si está rapidfuzz, mejor; si no, usa difflib.
        rows = []
        try:
            from rapidfuzz import process, fuzz
            scorer = fuzz.token_set_ratio
            emp_choices = list(emp_norm.keys())
            for c, nc in comp_norm.items():
                best = process.extractOne(nc, emp_choices, scorer=scorer)
                if best is None:
                    continue
                best_norm, score, _ = best
                e_orig = inv_emp.get(best_norm, [None])[0]
                rows.append({
                    "Company": c,
                    "Company_norm": nc,
                    "EMPRESA_best": e_orig,
                    "EMPRESA_norm": best_norm,
                    "score": float(score),
                })
            engine = "rapidfuzz"
        except Exception:
            from difflib import SequenceMatcher

            def ratio(a, b):
                return SequenceMatcher(None, a, b).ratio() * 100

            emp_choices = list(emp_norm.keys())
            for c, nc in comp_norm.items():
                best_norm = None
                best_score = -1
                for en in emp_choices:
                    sc = ratio(nc, en)
                    if sc > best_score:
                        best_score = sc
                        best_norm = en
                e_orig = inv_emp.get(best_norm, [None])[0] if best_norm is not None else None
                rows.append({
                    "Company": c,
                    "Company_norm": nc,
                    "EMPRESA_best": e_orig,
                    "EMPRESA_norm": best_norm,
                    "score": float(best_score),
                })
            engine = "difflib"

        df_fuzzy = pd.DataFrame(rows).sort_values('score', ascending=False).reset_index(drop=True)
        print(f"\nMotor de similitud: {engine}")
        print("Sugerencia: revisar coincidencias con score alto (p.ej. >= 90).")

        display(df_fuzzy.head(30))
        umbral = 90
        df_high = df_fuzzy[df_fuzzy['score'] >= umbral]
        print(f"\nCoincidencias sugeridas con score >= {umbral}: {len(df_high)}")
        display(df_high.reset_index(drop=True))

Coincidencias EXACTAS (tras normalizar): 0

Motor de similitud: difflib
Sugerencia: revisar coincidencias con score alto (p.ej. >= 90).


,Company,Company_norm,EMPRESA_best,EMPRESA_norm,score
0,OSRAM,OSRAM,CORSAM,CORSAM,72.727273
1,FEMSA,FEMSA,FADESA,FADESA,72.727273
2,SMI,SMI,BMI,BMI,66.666667
3,UCB,UCB,UPC,UPC,66.666667
4,BIC,BIC,BAC,BAC,66.666667
5,Chronos,CHRONOS,CONSEP,CONSEP,61.538462
6,BACARDI,BACARDI,BANRED,BANRED,61.538462
7,LACOSTE,LACOSTE,CONSEP,CONSEP,61.538462
8,Onapis,ONAPIS,AVIS,AVIS,60.000000
9,PEPSICO,PEPSICO,SIC,SIC,60.000000



Coincidencias sugeridas con score >= 90: 0


,Company,Company_norm,EMPRESA_best,EMPRESA_norm,score


### Interpretación técnica de esta validación

Si esta sección produce:

- **0 coincidencias exactas tras normalización**, y
- **0 coincidencias con score alto**,

entonces existe evidencia suficiente para afirmar que ambas fuentes no pueden integrarse de forma confiable mediante el nombre de la empresa.

Este resultado no representa una falla del código, sino un hallazgo del análisis de calidad e integración de datos.


> Nota de organización: la generación de **data cruda normalizada** (staging) y el **export** a CSV se movieron a `02_data_cleaning/01_raw_normalizado_export.ipynb` para no mezclar ingesta/profiling con cleaning.

## 7. Resultado de la validación

No se encontraron coincidencias exactas entre las empresas de ambas fuentes.

### Interpretación

Esto indica que:

- Las fuentes no están alineadas
- No es posible realizar un join directo confiable
- Existe inconsistencia en la representación de entidades

### Conclusión

Se rechaza la hipótesis de integración directa.

## 8. Próximos pasos (según evidencia)

- No forzar join `Company` ↔ `EMPRESA` por nombre: el profiling muestra que no hay coincidencias confiables.
- Preparar staging normalizado para BD/joins posteriores en `02_data_cleaning/01_raw_normalizado_export.ipynb` (export a CSV).
- Definir estrategia alternativa de integración (p.ej. catálogo unificado de empresas / matching asistido / llaves adicionales).